# 03 - Desarrollo y Comparación de Modelos ML
## Sistema Inteligente de Predicción de Riesgo para Deportistas Amateurs

**Objetivo:** Entrenar, optimizar y comparar 4 modelos de ML para predecir el riesgo de lesión. Seleccionar el mejor y serializarlo para la app Streamlit.

**Modelos:** Regresión Logística · Random Forest · XGBoost · Ensemble (Voting Classifier)

## 1. Importaciones y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve, classification_report)
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', palette='husl')
np.random.seed(42)

# Cargar splits
train = pd.read_csv('../data/processed/train.csv')
val   = pd.read_csv('../data/processed/validation.csv')
test  = pd.read_csv('../data/processed/test.csv')

X_train = train.drop(columns=['Injury_Indicator'])
y_train = train['Injury_Indicator']
X_val   = val.drop(columns=['Injury_Indicator'])
y_val   = val['Injury_Indicator']
X_test  = test.drop(columns=['Injury_Indicator'])
y_test  = test['Injury_Indicator']

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Features: {X_train.shape[1]}')

## 2. Función auxiliar de evaluación

Creamos una función reutilizable que calcula todas las métricas para cualquier modelo.

In [ ]:
def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_v, y_v):
    """Entrena el modelo y devuelve un dict con todas las métricas."""
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_v)
    y_prob = modelo.predict_proba(X_v)[:, 1]
    
    metricas = {
        'Modelo':     nombre,
        'Accuracy':   round(accuracy_score(y_v, y_pred), 4),
        'Precision':  round(precision_score(y_v, y_pred), 4),
        'Recall':     round(recall_score(y_v, y_pred), 4),
        'F1':         round(f1_score(y_v, y_pred), 4),
        'ROC-AUC':    round(roc_auc_score(y_v, y_prob), 4),
    }
    return metricas

resultados = []
print('Función de evaluación lista.')

## 3. Modelo 1: Regresión Logística (Baseline)

El modelo más simple. Nos da el punto de partida mínimo que deben superar los modelos más complejos.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)

# Validación cruzada 5-fold sobre train
cv_scores = cross_val_score(lr, X_train, y_train, cv=5, scoring='roc_auc')
print(f'Regresión Logística - CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

# Evaluar en validación
metricas_lr = evaluar_modelo('Regresión Logística', lr, X_train, y_train, X_val, y_val)
resultados.append(metricas_lr)

print('\nMétricas en validación:')
for k, v in metricas_lr.items():
    if k != 'Modelo':
        print(f'  {k}: {v}')

## 4. Modelo 2: Random Forest

Ensemble de árboles de decisión. Robusto ante overfitting y proporciona feature importance directamente.

In [ ]:
# GridSearch para optimizar hiperparámetros
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
}

rf_base = RandomForestClassifier(random_state=42, class_weight='balanced')
grid_rf = GridSearchCV(rf_base, param_grid_rf, cv=5, scoring='roc_auc', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print(f'Mejores parámetros RF: {grid_rf.best_params_}')
print(f'Mejor ROC-AUC CV: {grid_rf.best_score_:.4f}')

rf = grid_rf.best_estimator_
metricas_rf = evaluar_modelo('Random Forest', rf, X_train, y_train, X_val, y_val)
resultados.append(metricas_rf)

print('\nMétricas en validación:')
for k, v in metricas_rf.items():
    if k != 'Modelo':
        print(f'  {k}: {v}')

## 5. Modelo 3: XGBoost

Gradient Boosting optimizado. Suele ser el modelo más potente en datasets pequeños y tabulares.

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
}

xgb_base = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)
grid_xgb = GridSearchCV(xgb_base, param_grid_xgb, cv=5, scoring='roc_auc', n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print(f'Mejores parámetros XGB: {grid_xgb.best_params_}')
print(f'Mejor ROC-AUC CV: {grid_xgb.best_score_:.4f}')

xgb = grid_xgb.best_estimator_
metricas_xgb = evaluar_modelo('XGBoost', xgb, X_train, y_train, X_val, y_val)
resultados.append(metricas_xgb)

print('\nMétricas en validación:')
for k, v in metricas_xgb.items():
    if k != 'Modelo':
        print(f'  {k}: {v}')

## 6. Modelo 4: Ensemble (Voting Classifier)

Combina los 3 modelos anteriores por votación ponderada (soft voting). Si un modelo falla, los otros compensan.

In [ ]:
ensemble = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(max_iter=1000, random_state=42)),
        ('rf',  rf),
        ('xgb', xgb),
    ],
    voting='soft'
)

metricas_ens = evaluar_modelo('Ensemble', ensemble, X_train, y_train, X_val, y_val)
resultados.append(metricas_ens)

print('Métricas Ensemble en validación:')
for k, v in metricas_ens.items():
    if k != 'Modelo':
        print(f'  {k}: {v}')

## 7. Comparación de modelos

In [ ]:
df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print('TABLA COMPARATIVA DE MODELOS (validación):')
print(df_resultados.to_string())

# Gráfico de barras comparativo
fig, ax = plt.subplots(figsize=(13, 5))
df_resultados.plot(kind='bar', ax=ax, edgecolor='white', linewidth=0.8)
ax.set_title('Comparación de métricas por modelo', fontsize=14, fontweight='bold')
ax.set_ylabel('Puntuación')
ax.set_xlabel('')
ax.set_xticklabels(df_resultados.index, rotation=15, ha='right')
ax.axhline(0.7, color='red', linestyle='--', linewidth=1.2, label='Objetivo 70%')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('../data/processed/mod_01_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()

mejor_modelo_nombre = df_resultados['ROC-AUC'].idxmax()
print(f'\nMejor modelo por ROC-AUC: {mejor_modelo_nombre} ({df_resultados.loc[mejor_modelo_nombre, "ROC-AUC"]:.4f})')

## 8. Matrices de confusión

In [ ]:
modelos = [
    ('Regresión Logística', lr),
    ('Random Forest', rf),
    ('XGBoost', xgb),
    ('Ensemble', ensemble),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (nombre, modelo) in zip(axes, modelos):
    y_pred = modelo.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Sin lesión','Con lesión'],
                yticklabels=['Sin lesión','Con lesión'])
    ax.set_title(nombre, fontweight='bold', fontsize=11)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')

plt.suptitle('Matrices de confusión (validación)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/mod_02_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Curvas ROC-AUC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colores_roc = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for (nombre, modelo), color in zip(modelos, colores_roc):
    y_prob = modelo.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc = roc_auc_score(y_val, y_prob)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{nombre} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatorio (AUC=0.5)')
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=12)
ax.set_title('Curvas ROC-AUC por modelo', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/mod_03_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Feature Importance (Random Forest + XGBoost)

In [ ]:
feature_names = X_train.columns.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (nombre, modelo) in zip(axes, [('Random Forest', rf), ('XGBoost', xgb)]):
    importancias = modelo.feature_importances_
    indices = np.argsort(importancias)[::-1][:15]
    top_features = [feature_names[i] for i in indices]
    top_importancias = importancias[indices]
    
    colores = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(top_features))]
    bars = ax.barh(range(len(top_features)), top_importancias[::-1], color=colores[::-1], edgecolor='white')
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features[::-1], fontsize=10)
    ax.set_title(f'Feature Importance - {nombre}', fontweight='bold', fontsize=12)
    ax.set_xlabel('Importancia')

plt.suptitle('Top 15 features más importantes para predecir lesión', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/mod_04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features más importantes (Random Forest):')
importancias_rf = rf.feature_importances_
indices_rf = np.argsort(importancias_rf)[::-1][:5]
for i, idx in enumerate(indices_rf, 1):
    print(f'  {i}. {feature_names[idx]}: {importancias_rf[idx]:.4f}')

## 11. Evaluación final en Test Set

El test set solo se usa una vez, al final, para obtener las métricas reales del modelo seleccionado.

In [ ]:
# Seleccionar el mejor modelo por ROC-AUC en validación
mejor_idx = df_resultados['ROC-AUC'].idxmax()
modelos_dict = dict(modelos)
mejor_modelo = modelos_dict[mejor_idx]

print(f'Modelo seleccionado: {mejor_idx}')
print('=' * 50)

y_pred_test = mejor_modelo.predict(X_test)
y_prob_test = mejor_modelo.predict_proba(X_test)[:, 1]

print(f'Accuracy  : {accuracy_score(y_test, y_pred_test):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_test):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_test):.4f}')
print(f'F1        : {f1_score(y_test, y_pred_test):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_prob_test):.4f}')
print()
print('Reporte completo:')
print(classification_report(y_test, y_pred_test, target_names=['Sin lesión','Con lesión']))

## 12. Serializar modelos

In [ ]:
# Guardar todos los modelos
joblib.dump(lr,       '../models/modelo_lr.pkl')
joblib.dump(rf,       '../models/modelo_rf.pkl')
joblib.dump(xgb,      '../models/modelo_xgb.pkl')
joblib.dump(ensemble, '../models/modelo_ensemble.pkl')
joblib.dump(mejor_modelo, '../models/mejor_modelo.pkl')

# Guardar nombre del mejor modelo
joblib.dump(mejor_idx, '../models/mejor_modelo_nombre.pkl')

print('Modelos serializados en models/:')
import os
for f in sorted(os.listdir('../models')):
    size = os.path.getsize(f'../models/{f}') / 1024
    print(f'  - {f} ({size:.1f} KB)')

## 13. Conclusiones

In [ ]:
print('=' * 60)
print('CONCLUSIONES DEL NOTEBOOK DE MODELOS')
print('=' * 60)

for _, row in df_resultados.reset_index().iterrows():
    objetivo = 'OK' if row['ROC-AUC'] >= 0.70 else 'NO'
    print(f"  [{objetivo}] {row['Modelo']:25s} | ACC: {row['Accuracy']:.3f} | AUC: {row['ROC-AUC']:.3f}")

print(f"""
MODELO SELECCIONADO PARA PRODUCCION: {mejor_idx}

GRAFICOS GENERADOS:
  - mod_01_comparacion.png   (tabla comparativa)
  - mod_02_confusion.png     (matrices de confusion)
  - mod_03_roc.png           (curvas ROC-AUC)
  - mod_04_feature_importance.png

SIGUIENTE PASO:
  app/app.py: demo Streamlit que carga el mejor modelo
  y permite introducir datos para obtener el score de riesgo.
""")